# 08 — Machine Memory Layer

## Purpose

This notebook implements the **Memory layer** of the Smart Manufacturing AI system.

The LLM layer currently explains an actionable machine event, but it does not automatically remember the machine's previous events. This notebook creates a persistent, structured **episodic memory** from the historical actionable events and their LLM maintenance reports.

### Architecture

```text
IoT Simulation
      ↓
Smart Twin
      ↓
AI Agents
      ↓
Orchestrator
      ↓
Actionable Events
      ↓
LLM Maintenance Reports
      ↓
08_Memory.ipynb
      ↓
Machine Memory
      ↓
Historical context for future LLM decisions
```

### What this notebook does

1. Loads `llm_maintenance_reports.csv`.
2. Optionally enriches the memory with additional fields from `orchestrated_runtime_events.csv`.
3. Creates one persistent memory record for each actionable machine event.
4. Calculates simple historical trends such as RUL change and failure-probability change.
5. Creates per-machine memory summaries.
6. Provides retrieval functions for previous machine history.
7. Builds a compact historical context that can later be supplied to the LLM.
8. Saves the persistent memory artifacts.

**Important:** Memory is different from RAG.

- **Memory** = what our system has observed about its own machines in the past.
- **RAG** = external knowledge such as manuals, maintenance procedures, and technical documents.

This notebook intentionally does not replace the deterministic ML/agent decisions.


> **Compatibility note:** Step 8 explicitly iterates over machine groups instead of relying on pandas `GroupBy.apply(..., include_groups=False)`. This keeps the code compatible with the pandas version in the Python 3.13 environment.


In [1]:
# ============================================
# STEP 1: IMPORTS AND PROJECT PATHS
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

print("Python environment ready.")
print("Current working directory:", Path.cwd())


Python environment ready.
Current working directory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/notebooks


In [2]:
# ============================================
# STEP 2: LOCATE PROJECT DATA
# ============================================

# The notebook is normally executed from the project root.
# These candidates also make the notebook work if Jupyter
# was started from inside the notebooks directory.

possible_roots = [
    Path.cwd(),
    Path.cwd().parent
]

reports_path = None
orchestrator_path = None

for root in possible_roots:
    candidate = root / "data" / "processed" / "llm_maintenance_reports.csv"
    if candidate.exists():
        reports_path = candidate
        break

for root in possible_roots:
    candidate = root / "data" / "processed" / "orchestrated_runtime_events.csv"
    if candidate.exists():
        orchestrator_path = candidate
        break

if reports_path is None:
    raise FileNotFoundError(
        "Could not find data/processed/llm_maintenance_reports.csv. "
        "Run 07_LLM_Agent.ipynb first."
    )

print("LLM reports:", reports_path)
print("Orchestrator events:", orchestrator_path if orchestrator_path else "Not found (optional)")


LLM reports: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/llm_maintenance_reports.csv
Orchestrator events: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/orchestrated_runtime_events.csv


In [3]:
# ============================================
# STEP 3: LOAD LLM MAINTENANCE REPORTS
# ============================================

llm_reports_df = pd.read_csv(reports_path)

print("Shape:", llm_reports_df.shape)
print("Columns:")
print(llm_reports_df.columns.tolist())

display(llm_reports_df.head(3))


Shape: (55, 12)
Columns:
['Machine_ID', 'Machine_Type', 'Cycle', 'Simulation_Fault', 'failure_probability', 'predicted_rul_cycles', 'overall_status', 'maintenance_priority', 'maintenance_decision', 'alert_level', 'alert_required', 'llm_report']


,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_priority,maintenance_decision,alert_level,alert_required,llm_report
0,ROBOT_007,Industrial_Robot,157,ACTUATOR_DEGRADATION,0.0002,49.12,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
1,ROBOT_007,Industrial_Robot,161,ACTUATOR_DEGRADATION,0.0003,48.37,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
2,ROBOT_007,Industrial_Robot,163,ACTUATOR_DEGRADATION,0.0002,48.41,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...


In [4]:
# ============================================
# STEP 4: VALIDATE SOURCE DATA
# ============================================

required_columns = [
    "Machine_ID",
    "Machine_Type",
    "Cycle",
    "Simulation_Fault",
    "failure_probability",
    "predicted_rul_cycles",
    "overall_status",
    "maintenance_priority",
    "maintenance_decision",
    "alert_level",
    "alert_required",
    "llm_report"
]

missing = [
    col for col in required_columns
    if col not in llm_reports_df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Source validation successful.")
print("Actionable events:", len(llm_reports_df))
print("Machines:", llm_reports_df["Machine_ID"].nunique())


Source validation successful.
Actionable events: 55
Machines: 7


In [5]:
# ============================================
# STEP 5: OPTIONAL ORCHESTRATOR ENRICHMENT
# ============================================

memory_source = llm_reports_df.copy()

if orchestrator_path is not None:
    orchestrator_df = pd.read_csv(orchestrator_path)

    # Only bring in useful columns that are not already present.
    preferred_extra_columns = [
        "trigger_reason",
        "agents_invoked",
        "maintenance_reason",
        "alert_message"
    ]

    available_extra_columns = [
        col for col in preferred_extra_columns
        if col in orchestrator_df.columns
    ]

    if available_extra_columns:
        lookup_columns = [
            "Machine_ID",
            "Cycle"
        ] + available_extra_columns

        orchestrator_lookup = (
            orchestrator_df[lookup_columns]
            .drop_duplicates(subset=["Machine_ID", "Cycle"])
        )

        memory_source = memory_source.merge(
            orchestrator_lookup,
            on=["Machine_ID", "Cycle"],
            how="left"
        )

        print("Orchestrator enrichment added:")
        print(available_extra_columns)
    else:
        print("No additional orchestrator columns were available.")
else:
    print("Orchestrator CSV not found. Continuing with LLM report data.")

print("Memory source shape:", memory_source.shape)


Orchestrator enrichment added:
['trigger_reason', 'agents_invoked', 'maintenance_reason', 'alert_message']
Memory source shape: (55, 16)


In [6]:
# ============================================
# STEP 6: SORT EVENTS INTO MACHINE HISTORY
# ============================================

memory_source["Cycle"] = pd.to_numeric(
    memory_source["Cycle"],
    errors="coerce"
)

memory_source["failure_probability"] = pd.to_numeric(
    memory_source["failure_probability"],
    errors="coerce"
)

memory_source["predicted_rul_cycles"] = pd.to_numeric(
    memory_source["predicted_rul_cycles"],
    errors="coerce"
)

memory_source = (
    memory_source
    .sort_values(["Machine_ID", "Cycle"])
    .reset_index(drop=True)
)

# Historical features: compare each event with the previous
# actionable event for the same machine.
grouped = memory_source.groupby("Machine_ID", group_keys=False)

memory_source["previous_cycle"] = grouped["Cycle"].shift(1)

memory_source["previous_failure_probability"] = grouped[
    "failure_probability"
].shift(1)

memory_source["previous_rul_cycles"] = grouped[
    "predicted_rul_cycles"
].shift(1)

memory_source["previous_status"] = grouped[
    "overall_status"
].shift(1)

memory_source["cycle_gap"] = (
    memory_source["Cycle"]
    - memory_source["previous_cycle"]
)

memory_source["failure_probability_change"] = (
    memory_source["failure_probability"]
    - memory_source["previous_failure_probability"]
)

memory_source["rul_change_cycles"] = (
    memory_source["predicted_rul_cycles"]
    - memory_source["previous_rul_cycles"]
)

memory_source["status_changed"] = (
    memory_source["overall_status"]
    != memory_source["previous_status"]
)

# The first historical event for each machine has no previous event.
memory_source["has_previous_memory"] = (
    memory_source["previous_cycle"].notna()
)

print("Historical trend features created.")
display(
    memory_source[
        [
            "Machine_ID",
            "Cycle",
            "failure_probability",
            "previous_failure_probability",
            "failure_probability_change",
            "predicted_rul_cycles",
            "previous_rul_cycles",
            "rul_change_cycles",
            "overall_status",
            "previous_status",
            "status_changed"
        ]
    ].head(10)
)


Historical trend features created.


,Machine_ID,Cycle,failure_probability,previous_failure_probability,failure_probability_change,predicted_rul_cycles,previous_rul_cycles,rul_change_cycles,overall_status,previous_status,status_changed
0,CNC_001,193,0.0002,NaN,NaN,47.87,NaN,NaN,WARNING,NaN,True
1,CNC_001,195,0.0002,0.0002,0.0000,49.58,47.87,1.71,WARNING,WARNING,False
2,CNC_001,221,0.0003,0.0002,0.0001,17.02,49.58,-32.56,CRITICAL,WARNING,True
3,CNC_001,223,0.7748,0.0003,0.7745,13.21,17.02,-3.81,CRITICAL,CRITICAL,False
4,CNC_001,224,0.4115,0.7748,-0.3633,13.86,13.21,0.65,CRITICAL,CRITICAL,False
5,CNC_001,225,0.1137,0.4115,-0.2978,16.49,13.86,2.63,CRITICAL,CRITICAL,False
6,CNC_001,229,0.9952,0.1137,0.8815,9.80,16.49,-6.69,CRITICAL,CRITICAL,False
7,COMP_003,218,0.0007,NaN,NaN,43.89,NaN,NaN,WARNING,NaN,True
8,COMP_003,222,0.0004,0.0007,-0.0003,46.68,43.89,2.79,WARNING,WARNING,False
9,COMP_003,224,0.0004,0.0004,0.0000,45.27,46.68,-1.41,WARNING,WARNING,False


In [7]:
# ============================================
# STEP 7: CREATE PERSISTENT MEMORY RECORDS
# ============================================

memory_df = memory_source.copy()

memory_df.insert(
    0,
    "memory_id",
    memory_df.apply(
        lambda row: f"{row['Machine_ID']}_CYCLE_{int(row['Cycle'])}",
        axis=1
    )
)

memory_df.insert(
    1,
    "memory_type",
    "maintenance_event"
)

memory_df["memory_text"] = memory_df.apply(
    lambda row: (
        f"Machine {row['Machine_ID']} "
        f"({row['Machine_Type']}) at cycle {int(row['Cycle'])}: "
        f"status={row['overall_status']}, "
        f"failure_probability={row['failure_probability']:.4f}, "
        f"RUL={row['predicted_rul_cycles']:.2f} cycles, "
        f"fault={row['Simulation_Fault']}, "
        f"maintenance_decision={row['maintenance_decision']}, "
        f"alert_level={row['alert_level']}."
    ),
    axis=1
)

print("Persistent memory records created.")
print("Memory records:", len(memory_df))
print("Unique machines:", memory_df["Machine_ID"].nunique())

display(
    memory_df[
        [
            "memory_id",
            "memory_type",
            "Machine_ID",
            "Cycle",
            "overall_status",
            "maintenance_decision",
            "memory_text"
        ]
    ].head(5)
)


Persistent memory records created.
Memory records: 55
Unique machines: 7


,memory_id,memory_type,Machine_ID,Cycle,overall_status,maintenance_decision,memory_text
0,CNC_001_CYCLE_193,maintenance_event,CNC_001,193,WARNING,SCHEDULE INSPECTION,Machine CNC_001 (CNC_Mill) at cycle 193: statu...
1,CNC_001_CYCLE_195,maintenance_event,CNC_001,195,WARNING,SCHEDULE INSPECTION,Machine CNC_001 (CNC_Mill) at cycle 195: statu...
2,CNC_001_CYCLE_221,maintenance_event,CNC_001,221,CRITICAL,IMMEDIATE MAINTENANCE,Machine CNC_001 (CNC_Mill) at cycle 221: statu...
3,CNC_001_CYCLE_223,maintenance_event,CNC_001,223,CRITICAL,IMMEDIATE MAINTENANCE,Machine CNC_001 (CNC_Mill) at cycle 223: statu...
4,CNC_001_CYCLE_224,maintenance_event,CNC_001,224,CRITICAL,IMMEDIATE MAINTENANCE,Machine CNC_001 (CNC_Mill) at cycle 224: statu...


In [8]:
# ============================================
# STEP 8: BUILD MACHINE-LEVEL MEMORY SUMMARY
# ============================================

# IMPORTANT:
# In recent pandas versions, using
# groupby(...).apply(..., include_groups=False)
# removes the grouping column (Machine_ID) from each
# group. The earlier version therefore raised:
#
#     KeyError: 'Machine_ID'
#
# We avoid that ambiguity by explicitly iterating over
# each machine group and using the group key itself.

machine_summaries = []

for machine_id, group in memory_df.groupby("Machine_ID", sort=True):

    group = group.sort_values("Cycle").reset_index(drop=True)

    first = group.iloc[0]
    latest = group.iloc[-1]

    machine_summaries.append({
        "Machine_ID": machine_id,
        "Machine_Type": latest["Machine_Type"],
        "memory_event_count": len(group),
        "first_recorded_cycle": int(first["Cycle"]),
        "latest_recorded_cycle": int(latest["Cycle"]),
        "latest_status": latest["overall_status"],
        "latest_maintenance_priority": latest["maintenance_priority"],
        "latest_maintenance_decision": latest["maintenance_decision"],
        "latest_alert_level": latest["alert_level"],
        "latest_failure_probability": latest["failure_probability"],
        "latest_rul_cycles": latest["predicted_rul_cycles"],
        "minimum_rul_seen": group["predicted_rul_cycles"].min(),
        "maximum_failure_probability_seen": group["failure_probability"].max(),
        "maintenance_events": int(
            group["maintenance_decision"]
            .astype(str)
            .str.contains(
                "MAINTENANCE|INSPECTION",
                case=False,
                regex=True
            )
            .sum()
        ),
        "alert_events": int(
            group["alert_required"]
            .fillna(False)
            .astype(bool)
            .sum()
        ),
        "fault_types_seen": ", ".join(
            sorted(
                group["Simulation_Fault"]
                .dropna()
                .astype(str)
                .unique()
            )
        )
    })

machine_memory_summary = pd.DataFrame(machine_summaries)

print(
    "Machine summaries created:",
    len(machine_memory_summary)
)

display(machine_memory_summary)


Machine summaries created: 7


,Machine_ID,Machine_Type,memory_event_count,first_recorded_cycle,latest_recorded_cycle,latest_status,latest_maintenance_priority,latest_maintenance_decision,latest_alert_level,latest_failure_probability,latest_rul_cycles,minimum_rul_seen,maximum_failure_probability_seen,maintenance_events,alert_events,fault_types_seen
0,CNC_001,CNC_Mill,7,193,229,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.9952,9.80,9.80,0.9952,7,7,BEARING_WEAR
1,COMP_003,Compressor,9,218,255,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.9918,9.39,9.39,0.9918,9,9,VIBRATION
2,MOTOR_004,Electric_Motor,9,173,217,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.9963,9.14,9.14,0.9963,9,9,ELECTRICAL
3,PRESS_006,Hydraulic_Press,8,202,238,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.9938,11.62,11.62,0.9938,8,8,PRESSURE
4,PUMP_002,Hydraulic_Pump,7,236,274,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.7442,13.36,13.36,0.7442,7,7,OVERHEATING
5,ROBOT_007,Industrial_Robot,9,157,195,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.7254,8.96,8.96,0.9656,9,9,ACTUATOR_DEGRADATION
6,TURB_005,Turbine,6,254,284,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,0.9917,15.33,15.33,0.9917,6,6,MISALIGNMENT


In [9]:
# ============================================
# STEP 9: MEMORY RETRIEVAL FUNCTION
# ============================================

def retrieve_machine_history(
    machine_id,
    current_cycle=None,
    top_k=5
):
    """
    Retrieve the most recent historical memory records
    for a machine.

    If current_cycle is supplied, only events strictly
    before that cycle are returned.
    """

    history = memory_df[
        memory_df["Machine_ID"] == machine_id
    ].copy()

    if current_cycle is not None:
        history = history[
            history["Cycle"] < current_cycle
        ]

    history = (
        history
        .sort_values("Cycle", ascending=False)
        .head(top_k)
        .sort_values("Cycle")
        .reset_index(drop=True)
    )

    return history


print("Memory retrieval function created.")


Memory retrieval function created.


In [10]:
# ============================================
# STEP 10: BUILD HISTORICAL CONTEXT FOR LLM
# ============================================

def build_memory_context(
    machine_id,
    current_cycle=None,
    top_k=5
):
    """
    Convert retrieved machine history into a compact
    textual context that can later be supplied to the LLM.
    """

    history = retrieve_machine_history(
        machine_id=machine_id,
        current_cycle=current_cycle,
        top_k=top_k
    )

    if history.empty:
        return "No previous machine history was found."

    lines = [
        f"Historical memory for {machine_id}:"
    ]

    for _, row in history.iterrows():

        line = (
            f"- Cycle {int(row['Cycle'])}: "
            f"status={row['overall_status']}; "
            f"failure_probability="
            f"{row['failure_probability']:.4f}; "
            f"RUL={row['predicted_rul_cycles']:.2f} cycles; "
            f"fault={row['Simulation_Fault']}; "
            f"maintenance="
            f"{row['maintenance_decision']}; "
            f"alert={row['alert_level']}."
        )

        lines.append(line)

    return "\n".join(lines)


print("Historical LLM context builder created.")


Historical LLM context builder created.


In [11]:
# ============================================
# STEP 11: TEST MEMORY RETRIEVAL
# ============================================

# Select a machine with multiple historical events.
test_machine = (
    memory_df["Machine_ID"]
    .value_counts()
    .index[0]
)

test_history = retrieve_machine_history(
    machine_id=test_machine,
    top_k=5
)

print("Test machine:", test_machine)
print("\nRetrieved historical events:")
display(
    test_history[
        [
            "Machine_ID",
            "Cycle",
            "overall_status",
            "failure_probability",
            "predicted_rul_cycles",
            "Simulation_Fault",
            "maintenance_decision",
            "alert_level"
        ]
    ]
)

print("\nLLM-ready historical context:")
print(
    build_memory_context(
        machine_id=test_machine,
        top_k=5
    )
)


Test machine: COMP_003

Retrieved historical events:


,Machine_ID,Cycle,overall_status,failure_probability,predicted_rul_cycles,Simulation_Fault,maintenance_decision,alert_level
0,COMP_003,248,WARNING,0.0007,20.90,VIBRATION,SCHEDULE INSPECTION,WARNING
1,COMP_003,251,CRITICAL,0.0004,19.85,VIBRATION,IMMEDIATE MAINTENANCE,CRITICAL
2,COMP_003,253,CRITICAL,0.8326,12.56,VIBRATION,IMMEDIATE MAINTENANCE,CRITICAL
3,COMP_003,254,CRITICAL,0.5747,9.74,VIBRATION,IMMEDIATE MAINTENANCE,CRITICAL
4,COMP_003,255,CRITICAL,0.9918,9.39,VIBRATION,IMMEDIATE MAINTENANCE,CRITICAL



LLM-ready historical context:
Historical memory for COMP_003:
- Cycle 248: status=WARNING; failure_probability=0.0007; RUL=20.90 cycles; fault=VIBRATION; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 251: status=CRITICAL; failure_probability=0.0004; RUL=19.85 cycles; fault=VIBRATION; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 253: status=CRITICAL; failure_probability=0.8326; RUL=12.56 cycles; fault=VIBRATION; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 254: status=CRITICAL; failure_probability=0.5747; RUL=9.74 cycles; fault=VIBRATION; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 255: status=CRITICAL; failure_probability=0.9918; RUL=9.39 cycles; fault=VIBRATION; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.


In [12]:
# ============================================
# STEP 12: TEST "CURRENT EVENT + MEMORY"
# ============================================

# Demonstrate the key reason for having memory:
# a current event can be paired with previous events
# from the same machine.

latest_event = (
    memory_df
    .sort_values("Cycle")
    .iloc[-1]
)

current_machine = latest_event["Machine_ID"]
current_cycle = latest_event["Cycle"]

historical_context = build_memory_context(
    machine_id=current_machine,
    current_cycle=current_cycle,
    top_k=5
)

print("CURRENT MACHINE EVENT")
print("======================")
print(
    f"Machine: {current_machine}"
)
print(
    f"Cycle: {int(current_cycle)}"
)
print(
    f"Status: {latest_event['overall_status']}"
)
print(
    f"Failure probability: "
    f"{latest_event['failure_probability']:.4f}"
)
print(
    f"RUL: "
    f"{latest_event['predicted_rul_cycles']:.2f} cycles"
)
print(
    f"Maintenance decision: "
    f"{latest_event['maintenance_decision']}"
)

print("\nHISTORICAL MEMORY")
print("=================")
print(historical_context)


CURRENT MACHINE EVENT
Machine: TURB_005
Cycle: 284
Status: CRITICAL
Failure probability: 0.9917
RUL: 15.33 cycles
Maintenance decision: IMMEDIATE MAINTENANCE

HISTORICAL MEMORY
Historical memory for TURB_005:
- Cycle 254: status=WARNING; failure_probability=0.0005; RUL=48.61 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 259: status=WARNING; failure_probability=0.0007; RUL=49.96 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 281: status=CRITICAL; failure_probability=0.6913; RUL=19.87 cycles; fault=MISALIGNMENT; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 282: status=CRITICAL; failure_probability=0.2237; RUL=19.78 cycles; fault=MISALIGNMENT; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 283: status=WARNING; failure_probability=0.0416; RUL=20.23 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.


In [13]:
# ============================================
# STEP 13: SAVE MEMORY ARTIFACTS
# ============================================

processed_dir = reports_path.parent

memory_path = processed_dir / "machine_memory.csv"
summary_path = processed_dir / "machine_memory_summary.csv"

memory_df.to_csv(
    memory_path,
    index=False
)

machine_memory_summary.to_csv(
    summary_path,
    index=False
)

print("Memory artifacts saved successfully.")
print("Memory:", memory_path)
print("Machine summary:", summary_path)


Memory artifacts saved successfully.
Memory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory.csv
Machine summary: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory_summary.csv


In [14]:
# ============================================
# STEP 14: VERIFY PERSISTENT MEMORY
# ============================================

if not memory_path.exists():
    raise FileNotFoundError(
        f"Memory file was not created: {memory_path}"
    )

if not summary_path.exists():
    raise FileNotFoundError(
        f"Summary file was not created: {summary_path}"
    )

reloaded_memory = pd.read_csv(memory_path)
reloaded_summary = pd.read_csv(summary_path)

print("Memory file exists: True")
print("Summary file exists: True")
print("Memory shape:", reloaded_memory.shape)
print("Summary shape:", reloaded_summary.shape)

if len(reloaded_memory) == len(memory_df):
    print("Memory verification successful.")

if len(reloaded_summary) == len(machine_memory_summary):
    print("Machine summary verification successful.")


Memory file exists: True
Summary file exists: True
Memory shape: (55, 28)
Summary shape: (7, 16)
Memory verification successful.
Machine summary verification successful.


In [15]:
# ============================================
# STEP 15: MEMORY LAYER SUMMARY
# ============================================

print("============================================")
print("MEMORY LAYER SUMMARY")
print("============================================")

print(
    "Total memory records:",
    len(memory_df)
)

print(
    "Unique machines:",
    memory_df["Machine_ID"].nunique()
)

print(
    "Machine summaries:",
    len(machine_memory_summary)
)

print("\nEvents stored per machine:")
display(
    memory_df["Machine_ID"]
    .value_counts()
    .rename_axis("Machine_ID")
    .reset_index(name="memory_event_count")
)

print("\nPersistent memory files:")
print(memory_path)
print(summary_path)

print("\nMemory layer completed successfully.")


MEMORY LAYER SUMMARY
Total memory records: 55
Unique machines: 7
Machine summaries: 7

Events stored per machine:


,Machine_ID,memory_event_count
0,COMP_003,9
1,MOTOR_004,9
2,ROBOT_007,9
3,PRESS_006,8
4,CNC_001,7
5,PUMP_002,7
6,TURB_005,6



Persistent memory files:
/home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory.csv
/home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory_summary.csv

Memory layer completed successfully.


# Final Memory Architecture

The project now has persistent machine memory:

```text
Current/previous runtime events
            +
LLM maintenance reports
            ↓
      Structured Memory
            ↓
   machine_memory.csv
            +
machine_memory_summary.csv
```

The retrieval function can provide the LLM with previous events from the same machine.

### Why this matters

Without memory:

```text
Current event → LLM → explanation
```

With memory:

```text
Current event
      +
Previous machine history
      ↓
     LLM
      ↓
Context-aware maintenance intelligence
```

### Next phase: RAG

Memory answers:

> "What happened to this machine before?"

RAG will answer:

> "What does our available technical/maintenance knowledge say about this machine, fault, or maintenance procedure?"

The next notebook should therefore be:

```text
09_RAG.ipynb
```

where we will create a document knowledge base, retrieve relevant maintenance information, and combine it with the current machine event and machine memory.
